# Lab01 — Cotizaciones Óptimas de un Formador de Mercado

Este notebook **solo importa funciones de `src/` y genera gráficas**.
Toda la lógica de modelo y simulación vive en `src/model.py` y `src/simulation.py`.

In [1]:
import sys
import pathlib

import numpy as np

ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.model import (
    BASE_PI_I,
    BASE_PI_L,
    BASE_S0,
    LIQUIDITY_INTERCEPT,
    LIQUIDITY_SLOPE,
    execution_prob,
    expected_loss_ask,
    expected_loss_bid,
    optimize_quotes,
    price_pdf,
)
from src.simulation import monte_carlo, run_regime
from src.plots import (
    plot_execution_probability,
    plot_inventory_paths,
    plot_loss_functions,
    plot_monte_carlo_totals,
    plot_pnl_distributions,
    plot_price_distribution,
    plot_sensitivity_pi_I,
)

SEED = 42
N_TRADES = 10_000
MC_RUNS = 1_000
MC_TRADES = 1_000
SENSITIVITY_PI_I = [0.1, 0.4, 0.7]

np.random.seed(SEED)

## 1. Distribución del precio verdadero $f(P)$

In [2]:
fig = plot_price_distribution(price_pdf, BASE_S0)
fig

<Figure size 880x550 with 1 Axes>

## 2. Optimización de Bid y Ask (caso base)

In [3]:
result = optimize_quotes(S0=BASE_S0, pi_I=BASE_PI_I, pi_L=BASE_PI_L)
print(f"Bid optimo:        {result['bid']:.2f}")
print(f"Ask optimo:        {result['ask']:.2f}")
print(f"Spread optimo:     {result['spread']:.2f}")
print(f"Utilidad esperada: {result['expected_utility']:.2f}")

Bid optimo:        16.45
Ask optimo:        23.43
Spread optimo:     6.98
Utilidad esperada: 0.84


## 3. Pérdida esperada frente a traders informados, por lado

In [4]:
A_range = np.linspace(BASE_S0, BASE_S0 + 10, 100)
B_range = np.linspace(0.5, BASE_S0, 100)
loss_ask = [expected_loss_ask(A) for A in A_range]
loss_bid = [expected_loss_bid(B) for B in B_range]

fig = plot_loss_functions(A_range, loss_ask, B_range, loss_bid)
fig

<Figure size 880x550 with 1 Axes>

## 4. Probabilidad de ejecución vs. spread respecto a $S_0$

In [5]:
zero_spread = LIQUIDITY_INTERCEPT / LIQUIDITY_SLOPE
spread_range = np.linspace(0, zero_spread + 1, 200)
exec_prob = execution_prob(spread_range)

fig = plot_execution_probability(spread_range, exec_prob, zero_spread)
fig

<Figure size 880x550 with 1 Axes>

## 5. Simulación de 10,000 trades bajo tres regímenes (PnL e inventario)

In [6]:
regimes = {
    "Optimo": (result["bid"], result["ask"]),
    "Estrecho": (19.75, 20.05),
    "Amplio": (18.40, 21.40),
}

pnl_by_regime = {}
inventory_by_regime = {}
for name, (bid, ask) in regimes.items():
    stats_regime = run_regime(N_TRADES, bid, ask, BASE_S0, BASE_PI_I, BASE_PI_L)
    pnl_by_regime[name] = stats_regime["pnl"]
    inventory_by_regime[name] = stats_regime["inventory_path"]
    print(
        f"{name:10s} Bid={bid:6.2f} Ask={ask:6.2f}  "
        f"PnL total={stats_regime['total_pnl']:10.2f}  PnL medio={stats_regime['mean_pnl']:7.4f}  "
        f"Inv final={stats_regime['final_inventory']:8.0f}  |Inv| max={stats_regime['max_abs_inventory']:8.0f}"
    )

Optimo     Bid= 16.45 Ask= 23.43  PnL total=   8069.23  PnL medio= 0.8069  Inv final=       3  |Inv| max=      47
Estrecho   Bid= 19.75 Ask= 20.05  PnL total=  -7063.64  PnL medio=-0.7064  Inv final=      43  |Inv| max=      74
Amplio     Bid= 18.40 Ask= 21.40  PnL total=   3338.33  PnL medio= 0.3338  Inv final=       2  |Inv| max=      49


In [7]:
fig = plot_pnl_distributions(pnl_by_regime)
fig

<Figure size 880x550 with 1 Axes>

In [8]:
fig = plot_inventory_paths(inventory_by_regime)
fig

<Figure size 880x550 with 1 Axes>

## 6. Monte Carlo: 1,000 corridas de 1,000 trades

In [9]:
totals_by_regime = {}
for name, (bid, ask) in regimes.items():
    totals = monte_carlo(MC_RUNS, MC_TRADES, bid, ask, BASE_S0, BASE_PI_I, BASE_PI_L)
    totals_by_regime[name] = totals
    print(f"{name:10s} media={totals.mean():10.2f}  std={totals.std():8.2f}")

Optimo     media=    841.59  std=   50.84
Estrecho   media=   -675.14  std=   43.70


Amplio     media=    327.38  std=   43.04


In [10]:
fig = plot_monte_carlo_totals(totals_by_regime)
fig

<Figure size 880x550 with 1 Axes>

## 7. Análisis de sensibilidad: spread óptimo vs. $\pi_I$

In [11]:
sensitivity_spreads = []
for pi_I in SENSITIVITY_PI_I:
    pi_L = 1.0 - pi_I
    result_pi_I = optimize_quotes(S0=BASE_S0, pi_I=pi_I, pi_L=pi_L)
    sensitivity_spreads.append(result_pi_I["spread"])
    print(f"pi_I={pi_I:4.2f}  Bid={result_pi_I['bid']:6.2f}  Ask={result_pi_I['ask']:6.2f}  Spread={result_pi_I['spread']:6.2f}")

pi_I=0.10  Bid= 16.71  Ask= 23.11  Spread=  6.40


pi_I=0.40  Bid= 16.45  Ask= 23.43  Spread=  6.98


pi_I=0.70  Bid= 16.01  Ask= 24.00  Spread=  7.99


In [12]:
fig = plot_sensitivity_pi_I(SENSITIVITY_PI_I, sensitivity_spreads)
fig

<Figure size 880x550 with 1 Axes>